# Multiagent Movie Recommender

### Import Libraries

In [1]:
import re
import os
import random
import nest_asyncio
import pandas as pd

from typing import List, Dict, Union
from langchain_redis import RedisVectorStore, RedisCache, RedisChatMessageHistory
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate
from crewai import Agent, Task, Crew, Process
from langgraph.graph import StateGraph, END

from credentials import GROQ_API_KEY, HF_TOKEN

In [2]:
# set tokens as environment variables
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GROQ_API_KEY"] = GROQ_API_KEY

nest_asyncio.apply()

### Create Redis Vector DB
This is used for:
* The vector store (storing movie embeddings for similarity search)
* The exact-match cache (caching LLM responses to avoid redundant API calls)
* The chat message history (persisting conversation context across turns)

In [3]:
import redis
from credentials import REDIS_HOST, REDIS_PORT, REDIS_PASSWORD

# Create Redis client
redis_client = redis.Redis(
  host=REDIS_HOST,
  port=REDIS_PORT,
  password=REDIS_PASSWORD
)

# Test connection
redis_client.ping()
print("Connected to Redis successfully!")

Connected to Redis successfully!


In [4]:
# Clear Redis database (optional)
redis_client.flushdb()

True

### Download Movie Dataset

In [5]:
import zipfile

# Download MovieLens dataset (small version for demonstration)
!wget https://files.grouplens.org/datasets/movielens/ml-latest-small.zip

with zipfile.ZipFile('ml-latest-small.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

movies_df = pd.read_csv('ml-latest-small/movies.csv') # Contains columns: movieId, title, and genres
ratings_df = pd.read_csv('ml-latest-small/ratings.csv') # Contains columns: userId, movieId, rating, timestamp


--2026-02-28 20:08:03--  https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.96.204
Connecting to files.grouplens.org (files.grouplens.org)|128.101.96.204|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 978202 (955K) [application/zip]
Saving to: 'ml-latest-small.zip.19'

     0K .......... .......... .......... .......... ..........  5%  590K 2s
    50K .......... .......... .......... .......... .......... 10% 1.16M 1s
   100K .......... .......... .......... .......... .......... 15% 48.2M 1s
   150K .......... .......... .......... .......... .......... 20% 37.9M 0s
   200K .......... .......... .......... .......... .......... 26% 1.21M 0s
   250K .......... .......... .......... .......... .......... 31% 47.1M 0s
   300K .......... .......... .......... .......... .......... 36% 42.1M 0s
   350K .......... .......... .......... .......... .......... 41% 38.6M 0s
   400K ........

### Setup Embedding Model, Vector DB, LLM Response cache, & Chat History

* `Embeddings` — Initializes a HuggingFace embedding model that converts text into numerical vectors via the HF Inference API (no local GPU/torch required).

* `Vector store` — Takes the 1,000 movie titles, converts each to an embedding vector, and stores them in Redis. This enables semantic similarity search — e.g., searching "funny adventure film" will find movies with semantically similar titles/metadata.

* `Cache` — Sets up an exact-match LLM response cache in Redis Vector DB. If the same prompt is sent twice, the cached response is returned instead of making a new API call (saves cost/time).

* `Chat history` — Persists the conversation history in Redis so the agents can remember previous turns in the recommendation dialogue.

In [6]:
redis_url = f"redis://default:{REDIS_PASSWORD}@{REDIS_HOST}:{REDIS_PORT}"

# Use a smaller sample to stay within Redis free-tier memory limits
sample_df = movies_df.head(1000)

# HuggingFaceEndpointEmbeddings calls the HF Inference API — no local tokenizers/torch needed.
embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    huggingfacehub_api_token=HF_TOKEN,
)

# Create RedisVectorStore from movie titles, with metadata for each movie (e.g. genres)
vector_store = RedisVectorStore.from_texts(
    texts=sample_df['title'].tolist(),
    metadatas=sample_df.to_dict('records'),
    embedding=embeddings,
    redis_url=redis_url,
    index_name="movie_recommendations"
)

# Create RedisCache for storing intermediate results and avoid redundant llm calls
cache = RedisCache(redis_url=redis_url)

# Create RedisChatMessageHistory for storing conversation history and enabling retrieval of past interactions
chat_history = RedisChatMessageHistory("movie_recommendations", redis_url=redis_url)
print("Vector store, cache, and chat history initialized successfully.")

Vector store, cache, and chat history initialized successfully.


### Create Three Agents

####  Initialises the LLM that all three CrewAI agents will use

In [7]:
from crewai import LLM

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    temperature=0.5,
    api_key=GROQ_API_KEY,
)
print("✓ LLM initialized (Groq Llama-3.3-70B-Versatile)")


✓ LLM initialized (Groq Llama-3.3-70B-Versatile)


#### Define a Retrival tool that all three CrewAI agents can call to search the Vector DB

This converts the query to an embedding vector and finds the 5 most semantically similar movie titles stored in Redis



In [8]:
from crewai.tools import tool

@tool("Movie Database Lookup")
def retriever_tool(query: str) -> str:
    """Use this tool to search for movies in the database based on titles or descriptions."""
    results = vector_store.similarity_search(query, k=5)
    return "\n".join(f"{i+1}. {doc.page_content}" for i, doc in enumerate(results))


In [9]:
# Create CrewAgent 1
preference_analyst = Agent(
    role='Preference Analyst',
    goal='Analyze user preferences based on their input and chat history',
    backstory='You are an expert in understanding user preferences for movies',
    tools=[retriever_tool],
    llm=llm,
    verbose=True
)

In [10]:
# Create CrewAgent 2
movie_matcher = Agent(
    role='Movie Matcher',
    goal='Find movies that match user preferences',
    backstory='You are an expert in matching user preferences to movies in the database',
    tools=[retriever_tool],
    llm=llm,
    verbose=True
)

In [11]:
# Create CrewAgent 3
recommendation_generator = Agent(
    role='Recommendation Generator',
    goal='Generate personalized movie recommendations',
    backstory='You are an expert in creating engaging and personalized movie recommendations',
    tools=[retriever_tool],
    llm=llm,
    verbose=True
)

### Define Agent Tasks 

* These define the three tasks that the CrewAI agents will execute sequentially. 
* In CrewAI's sequential process, the output of one task is passed as input context to the next agent's task.

In [51]:
analyze_preferences_task = Task(
    description=(
        "Analyze the following user request and conversation history to identify their movie preferences.\n\n"
        "User request: {user_input}\n"
        "Chat history: {chat_history}"
    ),
    agent=preference_analyst,
    expected_output="A detailed analysis of the user's movie preferences"
)

match_movies_task = Task(
    description=(
        "Find movies that match the analyzed user preferences.\n\n"
        "Original user request: {user_input}"
    ),
    agent=movie_matcher,
    expected_output="A list of movies matching the user's preferences"
)

generate_recommendations_task = Task(
    description=(
        "Generate personalized movie recommendations based on the matched movies.\n\n"
        "Original user request: {user_input}"
    ),
    agent=recommendation_generator,
    expected_output="A personalized list of movie recommendations"
)


### Assemble Agents into a Crew

`crew` is CrewAI's top-level orchestrator that bundles agents and tasks together into a coordinated pipeline and manages their execution. Specifically, it:

* Manages agent collaboration — who does what and in what order
* Coordinates the three agents as a team to complete a multi-step task
* Operates inside LangGraph as a single node (run_crew)

In [52]:
# define top-level orchestrator
movie_crew = Crew(
    agents=[preference_analyst, movie_matcher, recommendation_generator], # registers the three specialist agents
    tasks=[analyze_preferences_task, match_movies_task, generate_recommendations_task], # defines the sequential execution order
    verbose=True # print each agent's internal reasoning 
)

### Setup LangGraph Workflow

LangGraph is the lower-level orchestration layer that wraps the CrewAI `crew`.
* The resulting graph is a simple one node, but the value is that LangGraph's StateGraph makes it easy to later extend. E.g., add a validation node, a retry branch, or split into parallel paths — without restructuring the whole system.

START → [run_crew] → END

##### (a.) Define the TypedDict classes for a LangGraph workflow

We defined 3 classes because LangGraph's StateGraph accepts three optional schema arguments:

* `UserInput` declares what data the caller must provide when invoking the graph (just user_input)
* `MovieState` is the combined internal state that flows through nodes (any node can read or write)
* `MovieOutput` declares what data the graph returns to the caller (just result)

This separation keeps the public API clean, callers don't need to know about internal state fields, and the graph's return value is predictably just the recommendation result.

In [53]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated

# Define the input type (i.e. what goes IN to the graph workflow)
class UserInput(TypedDict):
    user_input: str

# Define the output type (i.e. what goes OUT to the graph workflow)
class MovieOutput(TypedDict):
    result: str

# Define Combined state schema (i.e. required positional argument for StateGraph)
class MovieState(TypedDict):
    user_input: str
    result: str

##### (b.) Define LangGraph Node Function

* This is a single node, LangGraph graph that takes user input, runs the full multi-agent CrewAI pipeline, and returns the result.

* It bridges LangGraph's state management with the CrewAI pipeline:





In [59]:
# Define the workflow — the node function
def run_crew(state):
    user_input = state['user_input'] # extract user input from the LangGraph's state dict
    raw_history = chat_history.messages # retrieve conversation history from RedisChatMessageHistory
    # Serialize to plain dicts — CrewAI only accepts str/int/float/bool/dict/list
    history = [
        {"type": "human" if isinstance(msg, HumanMessage) else "ai", "content": msg.content}
        for msg in raw_history
    ]
    result = movie_crew.kickoff(inputs={'user_input': user_input, 'chat_history': history}) # Triggers the CrewAI crew to run all three agents sequentially (passing both the user input and chat history as inputs)
    return {"result": result} # writes the crew's final recommendation back into LangGraph's state under the result key, which the MovieOutput schema then exposes as the graph's output


#### (c.)  Instantiates LangGraph state machine with the TypedDict classes

In [60]:
# Create the workflow: instantiates the LangGraph state machine with a typed contract for what data can flow through it
workflow = StateGraph(
    MovieState,
    input_schema=UserInput,
    output_schema=MovieOutput
)

#### (d.) Add Node, Set Entrypoint, Add Edge

* Register the run_crew function as a node named "run_crew" in the graph. This is where the CrewAI pipeline executes.

* Tell LangGraph to start execution at the "run_crew" node when the graph is invoked.

*  Adds a directed edge from "run_crew" to the built-in END sentinel, meaning after run_crew finishes, the graph terminates.

In [61]:
from langgraph.graph import START

# Add the node
workflow.add_node("run_crew", run_crew)

# Set the entrypoint
workflow.add_edge(START, "run_crew")

# Add the edge to end the workflow
workflow.add_edge("run_crew", END)


#### (e.) Execute full pipeline
This triggers the whole pipeline chain:

LangGraph → routes input to run_crew → CrewAI runs the 3 specialized agents in sequence (analyze → match → recommend) → result is returned.

In [62]:
# Compile the workflow
app = workflow.compile()

#### Run an interactive loop

In [66]:
while True:
    user_input = input("What kind of movie are you in the mood for? (or 'quit' to exit): ")
    if user_input.lower() in ('quit', 'exit'):
        print("Thank you for using our movie recommendation system!")
        break

    result = app.invoke({"user_input": user_input})
    recommendation_text = str(result["result"])

    print(f"\nRecommendation:\n{recommendation_text}\n")

    # Persist turn to chat history for context in future turns
    chat_history.add_user_message(user_input)
    chat_history.add_ai_message(recommendation_text)

╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 7f1db7a4-2624-4dc7-b3fc-9fbd33790e6d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following user request and conversation history to identify their movie preferences.         │
│                                                                                                                 │
│  User request: I am interested in super hero movies                                                             │
│  Chat history: [{'type': 'human', 'content': 'I am interested in a nigerian movie'}, {'type': 'ai', 'content':  │
│  "Here's a list of Nigerian movie recommendations for you:\n1. Half of a Yellow Sun (2013)\n2. The Wedding      │
│  Party (2016)\n3. October 1 (2014)\n4. 30 Days in Atlanta (2014)\n5. A Trip to Jamaica (2016)"}, {'type':       │
│  'human', 'content': 'I am interested in indian movies'}, {'type': 'ai', 'content': "I think you'll love these  │
│  Nigerian movies. Here are some personalized recommendations just for you:\n\n1. Half of a Yellow Sun (2013) -  │
│  a historical drama that explores the Biafran war\n2. The Wedding Party (2016) - a romantic comedy that         │
│  showcases Nigerian weddings\n3. October 1 (2014) - a historical thriller that examines the events leading up   │
│  to Nigeria's independence\n4. 30 Days in Atlanta (2014) - a comedy that follows a Nigerian man's adventures    │
│  in the US\n5. A Trip to Jamaica (2016) - a comedy that explores cultural differences between Nigeria and       │
│  Jamaica\n6. Phone Swap (2012) - a romantic comedy that highlights the importance of communication in           │
│  relationships\n7. Lionheart (2018) - a drama that showcases a woman's journey to save her father's             │
│  business\n8. King of Boys (2018) - a crime thriller that explores the life of a powerful woman in Nigerian     │
│  society\n9. Mokalik (2019) - a drama that follows an 11-year-old boy's experiences as an apprentice in a       │
│  mechanic shop\n10. Beirut (2018) - an action thriller that explores the life of a diplomat in                  │
│  Lebanon\n\nThese movies offer a mix of drama, comedy, and action, and are all made in Nigeria or feature       │
│  Nigerian stories and culture. I hope you find something that interests you and enjoy watching these            │
│  movies!"}, {'type': 'human', 'content': 'do you have any action movies'}, {'type': 'ai', 'content': "I'm       │
│  excited to recommend some amazing Nigerian movies that I think you'll love. Based on your interest in          │
│  Nigerian cinema, I've curated a list of films that showcase the country's rich culture, language, and themes.  │
│  Here are my top picks:\n\n1. Half of a Yellow Sun (2013) - a historical drama that explores the Biafran war    │
│  and its impact on Nigerian society.\n2. The Wedding Party (2016) - a romantic comedy that delves into the      │
│  complexities of Nigerian weddings and the cultural nuances that come with them.\n3. October 1 (2014) - a       │
│  historical thriller that examines the events leading up to Nigeria's independence and the struggles that       │
│  shaped the nation.\n4. Lionheart (2018) - a comedy-drama film that showcases the Nigerian film industry and    │
│  explores themes of family, culture, and identity.\n5. 30 Days in Atlanta (2014) - a comedy that follows a      │
│  Nigerian man's adventures in the US and his experiences with cultural differences.\n6. A Trip to Jamaica       │
│  (2016) - a comedy that explores the cultural differences between Nigeria and Jamaica, and the hilarious        │
│  misadventures that ensue.\n7. Phone Swap (2012) - a romantic comedy that highlights the importance of          │
│  communication in relationships and the challenges of n

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Preference Analyst                                                                                      │
│                                                                                                                 │
│  Task: Analyze the following user request and conversation history to identify their movie preferences.         │
│                                                                                                                 │
│  User request: I am interested in super hero movies                                                             │
│  Chat history: [{'type': 'human', 'content': 'I am interested in a nigerian movie'}, {'type': 'ai', 'content':  │
│  "Here's a list of Nigerian movie recommendations for you:\n1. Half of a Yellow Sun (2013)\n2. The Wedding      │
│  Party (2016)\n3. October 1 (2014)\n4. 30 Days in Atlanta (2014)\n5. A Trip to Jamaica (2016)"}, {'type':       │
│  'human', 'content': 'I am interested in indian movies'}, {'type': 'ai', 'content': "I think you'll love these  │
│  Nigerian movies. Here are some personalized recommendations just for you:\n\n1. Half of a Yellow Sun (2013) -  │
│  a historical drama that explores the Biafran war\n2. The Wedding Party (2016) - a romantic comedy that         │
│  showcases Nigerian weddings\n3. October 1 (2014) - a historical thriller that examines the events leading up   │
│  to Nigeria's independence\n4. 30 Days in Atlanta (2014) - a comedy that follows a Nigerian man's adventures    │
│  in the US\n5. A Trip to Jamaica (2016) - a comedy that explores cultural differences between Nigeria and       │
│  Jamaica\n6. Phone Swap (2012) - a romantic comedy that highlights the importance of communication in           │
│  relationships\n7. Lionheart (2018) - a drama that showcases a woman's journey to save her father's             │
│  business\n8. King of Boys (2018) - a crime thriller that explores the life of a powerful woman in Nigerian     │
│  society\n9. Mokalik (2019) - a drama that follows an 11-year-old boy's experiences as an apprentice in a       │
│  mechanic shop\n10. Beirut (2018) - an action thriller that explores the life of a diplomat in                  │
│  Lebanon\n\nThese movies offer a mix of drama, comedy, and action, and are all made in Nigeria or feature       │
│  Nigerian stories and culture. I hope you find something that interests you and enjoy watching these            │
│  movies!"}, {'type': 'human', 'content': 'do you have any action movies'}, {'type': 'ai', 'content': "I'm       │
│  excited to recommend some amazing Nigerian movies that I think you'll love. Based on your interest in          │
│  Nigerian cinema, I've curated a list of films that showcase the country's rich culture, language, and themes.  │
│  Here are my top picks:\n\n1. Half of a Yellow Sun (2013) - a historical drama that explores the Biafran war    │
│  and its impact on Nigerian society.\n2. The Wedding Party (2016) - a romantic comedy that delves into the      │
│  complexities of Nigerian weddings and the cultural nuances that come with them.\n3. October 1 (2014) - a       │
│  historical thriller that examines the events leading up to Nigeria's independence and the struggles that       │
│  shaped the nation.\n4. Lionheart (2018) - a comedy-drama film that showcases the Nigerian film industry and    │
│  explores themes of family, culture, and identity.\n5. 30 Days in Atlanta (2014) - a comedy that follows a      │
│  Nigerian man's adventures in the US and his experiences with cultural differences.\n6. A Trip to Jamaica       │
│  (2016) - a comedy that explores the cultural differences between Nigeria and Jamaica, and the hilarious        │
│  misadventures that ensue.\n7. Phone Swap (2012) - a ro

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Preference Analyst                                                                                      │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Based on the user's request, "I am interested in super hero movies," it is clear that they are looking for     │
│  films that feature superheroes, action, and adventure. The user's conversation history reveals that they have  │
│  previously expressed interest in a wide range of genres, including Nigerian movies, Indian movies, action      │
│  movies, Christmas movies, horror movies, romance movies, and funny movies.                                     │
│                                                                                                                 │
│  The user's initial request for Nigerian movies suggests that they may be interested in films that showcase     │
│  Nigerian culture, language, or themes. The subsequent requests for Indian movies, action movies, horror        │
│  movies, romance movies, and funny movies indicate that they may also be interested in exploring films from     │
│  other international cultures and genres.                                                                       │
│                                                                                                                 │
│  The request for superhero movies suggests that the user may enjoy films that feature action, adventure, and    │
│  thrilling sequences. Some examples of superhero movies that the user might enjoy include:                      │
│                                                                                                                 │
│  1. "The Avengers" (2012) - a superhero film that brings together some of Marvel's most iconic heroes,          │
│  including Iron Man, Captain America, and Thor.                                                                 │
│  2. "The Dark Knight" (2008) - a superhero film that tells the story of Batman's battle against the Joker,      │
│  featuring a strong protagonist and a gripping narrative.                                                       │
│  3. "Wonder Woman" (2017) - a superhero film that tells the story of the iconic DC Comics heroine, featuring a  │
│  strong female lead and a mix of action, adventure, and humor.                                                  │
│  4. "Black Panther" (2018) - a superhero film that tells the story of the king of Wakanda, featuring a diverse  │
│  cast and a mix of action, adventure, and social commentary.                                                    │
│  5. "Spider-Man: Into the Spider-Verse" (2018) - an animated superhero film that tells the story of Miles       │
│  Morales, a teenager who becomes the new Spider-Man, featuring a unique visual style and a mix of action,       │
│  adventure, and humor.                                                                                          │
│                                                                                                                 │
│  The user's interest in superhero movies and their previous requests for other genres suggest that they may     │
│  enjoy films that combine elements of action, adventure, and thrilling sequences with a mix of genres and       │
│  themes. Some examples of such films include:                                                                   │
│                                                                                                                 │
│  1. "The Matrix" (1999) - a science fiction film that f

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Generate personalized movie recommendations based on the matched movies.                                 │
│                                                                                                                 │
│  Original user request: I am interested in super hero movies                                                    │
│  Agent: Recommendation Generator                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 7f1db7a4-2624-4dc7-b3fc-9fbd33790e6d                                                                       │
│  Final Output: I'm excited to recommend some fantastic superhero movies that I think you'll love. Based on      │
│  your interest in superhero movies and your previous requests for Nigerian movies, Indian movies, action        │
│  movies, Christmas movies, horror movies, romance movies, and funny movies, I've curated a list of films that   │
│  combine elements of action, adventure, and thrilling sequences with a mix of genres and themes. Here are my    │
│  top picks:                                                                                                     │
│                                                                                                                 │
│  1. The Avengers (2012) - a superhero film that brings together some of Marvel's most iconic heroes, including  │
│  Iron Man, Captain America, and Thor.                                                                           │
│  2. The Dark Knight (2008) - a superhero film that tells the story of Batman's battle against the Joker,        │
│  featuring a strong protagonist and a gripping narrative.                                                       │
│  3. Wonder Woman (2017) - a superhero film that tells the story of the iconic DC Comics heroine, featuring a    │
│  strong female lead and a mix of action, adventure, and humor.                                                  │
│  4. Black Panther (2018) - a superhero film that tells the story of the king of Wakanda, featuring a diverse    │
│  cast and a mix of action, adventure, and social commentary.                                                    │
│  5. Spider-Man: Into the Spider-Verse (2018) - an animated superhero film that tells the story of Miles         │
│  Morales, a teenager who becomes the new Spider-Man, featuring a unique visual style and a mix of action,       │
│  adventure, and humor.                                                                                          │
│  6. The Matrix (1999) - a science fiction film that features a mix of action, adventure, and thrilling          │
│  sequences, with a strong protagonist and a thought-provoking narrative.                                        │
│  7. Inception (2010) - a science fiction film that features a mix of action, adventure, and thrilling           │
│  sequences, with a strong protagonist and a complex narrative.                                                  │
│  8. Interstellar (2014) - a science fiction film that features a mix of action, adventure, and thrilling        │
│  sequences, with a strong protagonist and a thought-provoking narrative.                                        │
│  9. **Mad Max**: Fury Road (2015) - an action film that features a mix of action, adventure, and thrilling      │
│  sequences, with a strong female lead and a gripping narrative.                                                 │
│  10. The Hunger Games (2012) - a science fiction film that features a mix of action, adventure, and thrilling   │
│  sequences, with a strong female lead and a thought-provoking narrative.                                        │
│  11. The Crossover (2018) - a Nigerian romantic comedy film that tells the story of a young man who falls in    │
│  love with a woman during the Easter season, featuring a mix of romance, comedy, and drama.                     │
│  12. Easter in Lagos (2019) - a Nigerian drama film th


Recommendation:
I'm excited to recommend some fantastic superhero movies that I think you'll love. Based on your interest in superhero movies and your previous requests for Nigerian movies, Indian movies, action movies, Christmas movies, horror movies, romance movies, and funny movies, I've curated a list of films that combine elements of action, adventure, and thrilling sequences with a mix of genres and themes. Here are my top picks:

1. The Avengers (2012) - a superhero film that brings together some of Marvel's most iconic heroes, including Iron Man, Captain America, and Thor.
2. The Dark Knight (2008) - a superhero film that tells the story of Batman's battle against the Joker, featuring a strong protagonist and a gripping narrative.
3. Wonder Woman (2017) - a superhero film that tells the story of the iconic DC Comics heroine, featuring a strong female lead and a mix of action, adventure, and humor.
4. Black Panther (2018) - a superhero film that tells the story of the king of W

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Thank you for using our movie recommendation system!


In [ ]:
# Clear the vector store
#vector_store.index.delete(drop=True)

# Clear the chat history
#chat_history.clear()